# RetailMind AI — Day 17: Voice Support and Multimodal Search

Build a local CLIP index for Fashion Product Images, search the catalog by text or image, and synthesize support replies as speech. This notebook uses `src/day17_voice_multimodal_search.py`.

Install `requirements.txt` first. The CLIP and TTS model checkpoints download from Hugging Face on first use. Index creation, search inference, and TTS are opt-in below. A full 44k-image catalog index needs substantial RAM and takes much longer on CPU; use a small `limit` for a first run.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display
from PIL import Image
from src.day17_voice_multimodal_search import (
    DEFAULT_DATA_DIR, DEFAULT_INDEX_DIR, DEFAULT_TTS_MODEL,
    load_catalog, build_catalog_index, search_by_text, search_by_image,
    TextToSpeechSpeaker, VoiceSupportAssistant,
)

ROOT = Path.cwd()
catalog_rows = load_catalog(DEFAULT_DATA_DIR, limit=5)
[(row['id'], row.get('productDisplayName'), row.get('articleType'), row['image_path']) for row in catalog_rows]

## 1. Build multimodal CLIP embeddings (opt in)

For a quick check, keep `INDEX_LIMIT` small. Set it to `None` to embed every product with a readable photo. The index stores normalized image and catalog-text embeddings in float16 under `models/day17_clip_catalog/` (ignored by Git).

In [ ]:
BUILD_INDEX = False
INDEX_LIMIT = 200  # set to None for all available products
if BUILD_INDEX:
    index_report = build_catalog_index(
        data_dir=DEFAULT_DATA_DIR, output_dir=DEFAULT_INDEX_DIR,
        batch_size=16, limit=INDEX_LIMIT,
    )
    index_report
else:
    print('Index build skipped. Set BUILD_INDEX = True to run CLIP over the catalog.')

## 2. Search by text or a query photo

Text search compares the query with both product-photo and metadata embeddings. Image search compares a query photo with the same shared CLIP space. Similarity is cosine similarity, not a calibrated probability.

In [ ]:
RUN_SEARCH = False
if RUN_SEARCH:
    text_results = search_by_text('navy blue casual shirt for men', top_k=5)
    [(item.get('productDisplayName'), item['similarity'], item['image_path']) for item in text_results]
else:
    print('Search skipped. Build or load an index, then set RUN_SEARCH = True.')

In [ ]:
if RUN_SEARCH:
    query_image = catalog_rows[0]['image_path']
    image_results = search_by_image(query_image, top_k=5)
    for item in image_results:
        print(item.get('productDisplayName'), round(item['similarity'], 3), item['image_path'])
    display(Image.open(query_image).convert('RGB'))
else:
    print('Image search skipped until the CLIP index is ready.')

## 3. Speak a support reply (opt in)

This cell synthesizes the provided reply without making an API call. To speak an actual Day 14 support answer, wrap a configured `RetailSupportAssistant` with `VoiceSupportAssistant`; that uses your `OPENAI_API_KEY` when the support response is requested.

In [ ]:
RUN_TTS = False
if RUN_TTS:
    speaker = TextToSpeechSpeaker(DEFAULT_TTS_MODEL)
    speech = speaker.speak(
        'Your Everyday Tote is made from cotton canvas and holds eighteen liters.',
        ROOT / 'reports' / 'day17_voice_reply.wav',
    )
    print(speech)
    display(Audio(filename=speech['output_path']))
else:
    print('Speech generation skipped. Set RUN_TTS = True to download the TTS model and create a WAV reply.')

## Review the index report and assets

The index report records the selected model, device, indexed product count, embedding dimensions, and build time. Search scores rank candidates and should be reviewed as a relevance aid, not treated as a confidence percentage. The generated WAV and local embedding cache are not committed to the repository.

## Saved output refresh

This cell displays the saved metrics and figures from `reports/` for this notebook. Training and generation cells are kept intact. Model-heavy workflows could not be rerun here because the required TensorFlow, PyTorch, or Transformers packages are unavailable; opt-in generation cells remain disabled.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, SVG, display

# Re-display the saved metrics and figures for this notebook's project day.
# Set ROOT to the repository root if the notebook is launched elsewhere.
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
REPORT_DIR = ROOT / "reports"
DAYS = [17]
print("Showing saved report artifacts for project day(s):", DAYS)
for day in DAYS:
    for path in sorted(REPORT_DIR.glob(f"day{day}*")):
        if path.is_dir():
            for image_path in sorted(path.rglob("*.png")):
                display(Image(filename=str(image_path)))
        elif path.suffix.lower() == ".csv":
            print("\n" + path.name)
            display(pd.read_csv(path, nrows=25))
        elif path.suffix.lower() == ".png":
            display(Image(filename=str(path)))
        elif path.suffix.lower() == ".svg":
            display(SVG(filename=str(path)))


Showing saved report artifacts for project day(s): [17]


No saved metric, image, or benchmark artifacts are available for these days. Day 17 CLIP indexing/search and TTS are opt-in and require PyTorch/Transformers model weights.
